# QUOPS with CUDA-Q

QUOPS measures the largest computationally relevant quantum circuit hardware can execute within a specified accuracy threshold and how quickly it can do so.


In [ ]:
# %pip install -r requirements.txt

import cudaq
import numpy as np
import pandas as pd
from time import perf_counter
from IPython.display import display
from src import *

cudaq.reset_target()
cudaq.set_target("density-matrix-cpu")
# cudaq.set_target("qpp-cpu")
# cudaq.set_target("nvidia", option="fp32") # gpu accelerated statevector simulator
# cudaq.set_target("nvidia", option="fp64") # Higher arithmetic precision and memory use.

# Setup: https://nvidia.github.io/cuda-quantum/0.16.0/using/backends/hardware.html
# cudaq.set_target("quantinuum", machine="H2-2", project="<NEXUS_PROJECT>")
# cudaq.set_target("ionq", qpu="qpu.aria-1")
# cudaq.set_target("iqm", url="https://<IQM_SERVER>/")
# cudaq.set_target("oqc", url="<OQC_URL>", machine="toshiko")
# cudaq.set_target("anyon", machine="telegraph-8q")
# cudaq.set_target("infleqtion", machine="cq_sqale_qpu")
# cudaq.set_target("braket", machine="arn:aws:braket:us-west-1::device/qpu/rigetti/Ankaa-3")
# cudaq.set_target("pasqal", machine="FRESNEL") # Analog: cudaq.evolve / cudaq.evolve_async.
# cudaq.set_target("quera") # QuEra Aquila via Braket; analog: cudaq.evolve / cudaq.evolve_async.
# cudaq.set_target("orca", url="<ORCA_SERVER_URL>") # Photonic boson sampling: cudaq.orca.sample.


# noise = None
noise = make_helios1_noise()
# noise = make_depolarizing_noise(p_1q=0.004, p_2q=0.02)

shots = 256
num_circuits = 100 # Independent circuits per shape
num_mirrors = 1 # Fresh mirror randomizations for each forward circuit
bootstrap_resamples = 2000 # Estimate bootstrap spread as a diagnostic.
familywise_alpha = 0.05 # Family error budget for Hochberg across all declared shapes.
seed = 2026
rc_style = "pauli_frame" # Randomize reference circuits while preserving their ideal unitary


## Circuit shape and mirror measurements

Layers of random CNOT pairings alternate with Pauli rotations, with axes and angles sampled independently and uniformly:

$$R_P(\theta)=e^{-i\theta P/2},\qquad P\in\{X,Y,Z\},\quad\theta\in[0,2\pi).$$

**Width** $w$ counts qubits. **Paper depth** $d$ counts elementary layers, so there are $d/2$ CNOT/rotation pairs. $\zeta$ is the active-qubit fraction in the final pair. **Forward size** counts qubit participations:

$$s=N_{R_P}+2N_{\rm CX}.$$

### BR, RR and REF

MCFE uses mirrors with known ideal output strings. With implementation $g$, randomized reference compiler $g_{\rm ref}$ and local-Clifford caps $L,L'$ (rightmost acts first),

$$
\begin{aligned}
M_{\rm BR}&=L'g_{\rm ref}(C^\dagger)g(C)L,\\
M_{\rm RR}&=L'g_{\rm ref}(C^\dagger)g_{\rm ref}(C)L,\\
M_{\rm REF}&=L'L.
\end{aligned}
$$

BR probes the circuit; RR and REF estimate reference and cap/readout contributions. If $\hat h_k$ is the fraction of shots at Hamming distance $k$ from a mirror's target, its effective polarization is

$$\hat\lambda=\frac{\sum_{k=0}^{w}(-1/2)^k\hat h_k-4^{-w}}{1-4^{-w}}.$$



In [ ]:
widths = (3, 4) #range of qubit counts 
depths = (4, 6) #range of depths 
zeta = 1.0 # Active-qubit fraction in the final pair.

# Keep only shapes inside the paper's utility cone: width**2 <= size <= width**3.
shapes = default_shape_schedule([{"width": w, "depth": d, "zeta": zeta} for w in widths for d in depths if in_utility_cone(w, shape_size(w, d, zeta))])

# Retain each circuit/mirror observation and update one live table in place.
rows = []
measurements = []
data = pd.DataFrame(rows)
live_table = display(data, display_id=True)

rng = np.random.default_rng(seed)
cudaq.set_random_seed(seed)

for shape in shapes:

    width, depth, size = shape["width"], shape["depth"], shape["size"]

    for circuit_index in range(num_circuits):

        # Draw the CNOT pairings, active subset, rotation axes and angles.
        C = sample_quops_circuit(width, depth, shape["zeta"], seed=rng)
        C_inverse = adjoint_C(C) # Reverse this exact sampled circuit.

        # Reuse C across its mirrors; independently drawn C circuits define the bootstrap clusters.
        for mirror_index in range(num_mirrors):

            # Independent caps give each mirror a known ideal target.
            br_caps = sample_mcfe_caps(width, seed=rng)
            L_br, L_prime_br, target_bits_br = br_caps.initial, br_caps.final, br_caps.target

            rr_caps = sample_mcfe_caps(width, seed=rng)
            L_rr, L_prime_rr, target_bits_rr = rr_caps.initial, rr_caps.final, rr_caps.target

            ref_caps = sample_mcfe_caps(width, seed=rng)
            L_ref, L_prime_ref, target_bits_ref = ref_caps.initial, ref_caps.final, ref_caps.target

            # Randomize each reference independently, preserving its ideal operation.
            rc_C_inverse_br = randomized_compilation(C_inverse, seed=rng, style=rc_style)
            rc_C_rr = randomized_compilation(C, seed=rng, style=rc_style)
            rc_C_inverse_rr = randomized_compilation(C_inverse, seed=rng, style=rc_style)

            # Time the BR and RR sampling calls for the simulator-rate proxy.
            start_time = perf_counter()
            br_counts = cudaq.sample(BR_kernel, width, L_br, C, rc_C_inverse_br, L_prime_br, shots_count=shots, noise_model=noise)
            rr_counts = cudaq.sample(RR_kernel, width, L_rr, rc_C_rr, rc_C_inverse_rr, L_prime_rr, shots_count=shots, noise_model=noise)
            br_rr_seconds = perf_counter() - start_time

            # Collect fresh independent REF data for each observation.
            ref_counts = cudaq.sample(REF_kernel, width, L_ref, L_prime_ref, shots_count=shots, noise_model=noise)

            # Convert all Hamming distances from the known target into each mirror's effective polarization.
            lambda_br = effective_polarization_from_counts(br_counts, width, target_bits_br)
            lambda_rr = effective_polarization_from_counts(rr_counts, width, target_bits_rr)
            lambda_ref = effective_polarization_from_counts(ref_counts, width, target_bits_ref)

            # Keep component estimates for ensemble inference and BR/RR shots plus timing for the rate.
            rows.append({"shape": f"w={width},s={size}", "width": width, "depth": depth, "size": size, "zeta": shape["zeta"], "circuit_index": circuit_index, "mirror_index": mirror_index, "lambda_br": lambda_br, "lambda_rr": lambda_rr, "lambda_ref": lambda_ref, "br_rr_seconds": br_rr_seconds, "m1_m2_shots": int(br_counts.get_total_shots() + rr_counts.get_total_shots()), "shots_br": int(br_counts.get_total_shots()), "shots_rr": int(rr_counts.get_total_shots()), "shots_ref": int(ref_counts.get_total_shots())})
            measurements.append({"width": width, "size": size, "circuit_index": circuit_index, "mirror_index": mirror_index, "counts": {"br": counts_dict(br_counts), "rr": counts_dict(rr_counts), "ref": counts_dict(ref_counts)}, "targets": {"br": target_bits_br, "rr": target_bits_rr, "ref": target_bits_ref}})
            data = pd.DataFrame(rows)
            live_table.update(data)

# Retain the complete acquisition and its raw evidence before inference.
evidence_path = save_notebook_evidence(data, measurements, shapes, {"shots": shots, "num_circuits": num_circuits, "num_mirrors": num_mirrors, "seed": seed, "bootstrap_resamples": bootstrap_resamples, "familywise_alpha": familywise_alpha, "multiplicity": "hochberg", "certification_method": "hoeffding", "rc_style": rc_style, "target": str(cudaq.get_target()), "noise": describe_noise(noise, max(widths))})
evidence_path


## Mean polarization and conservative shape tests

This notebook uses conservative Hoeffding–Hochberg inference, an alternative to the statistical procedure used in the paper’s experiments.

After acquisition finishes, the code calculates five quantities for each shape:

1. **Average lambdas:** average the BR, RR and REF polarizations separately.
2. **Gamma:** calculate the ratio of means, $\hat\gamma_{w,s}=\bar\lambda_{\mathrm{BR}}/\sqrt{\bar\lambda_{\mathrm{RR}}\bar\lambda_{\mathrm{REF}}}$.
3. **Sigma:** estimate the bootstrap standard deviation of gamma. Resample paired BR/RR circuit groups and independent REF observations. Sigma is a diagnostic.
4. **Gamma lower:** calculate an individual conservative 95% lower bound at the default settings. Hoeffding uses component means and independent sample counts from the sigma table. Sigma's numerical value does not set the bound.
5. **P-values:** use the same conservative bounds to test whether the population MCFE ratio exceeds $1/\sqrt e$.

**Final pass rule:** `hochberg_test()` adjusts the p-values across **all declared shapes**. A shape passes when `test_valid` is true and `adjusted_p_value <= familywise_alpha` (default 0.05). Missing or invalid tests count as p = 1 during adjustment and cannot pass.

Comparing `gamma_lower > 1/sqrt(e)` is the individual, unadjusted check. The plot and the largest certified tested size use the **Hochberg decisions**, not that individual check. No gated-score procedure is applied. A non-pass means the threshold was not established; it does not prove the shape is below threshold.

Declare the shape family before acquisition; Hochberg here assumes independent data across shapes. The bounds require independent original circuits, independent REF observations and positive population RR/REF means. More mirrors do not add independent original circuits. Interpreting the ratio as process polarization still requires the MCFE model and reference-compiler assumptions. See `calculate_gamma_lower()` in [src.py](src.py) for the bound.

**Simulator throughput** is calculated separately as $2s\hat\gamma_{w,s}^{2}N/T$, using summed BR/RR shots $N$ and sampling-call time $T$. REF and work outside those calls are excluded.


In [ ]:
# Average the BR, RR and REF observations within each tested shape.
avg_lambdas_for_each_shape = calculate_mean_lambdas(data)

# Normalize the mean lambdas to estimate each shape's mean process polarization.
gamma_for_each_shape = calculate_gamma_hat_ws(avg_lambdas_for_each_shape)

# Estimate bootstrap sigma and retain independent circuit/reference counts for the bounds.
sigma_for_each_shape = calculate_sigma(data, bootstrap_resamples, seed)

# Calculate an individual conservative lower bound; the sigma value does not set this bound.
gamma_lower_for_each_shape = calculate_gamma_lower(gamma_for_each_shape, sigma_for_each_shape, alpha=familywise_alpha)

# Calculate conservative one-sided p-values for the threshold 1/sqrt(e).
p_value_for_each_shape = calculate_p_value(gamma_for_each_shape, sigma_for_each_shape)

# Final passes require a valid test and adjusted_p_value <= familywise_alpha.
hochberg_for_each_shape = hochberg_test(p_value_for_each_shape, alpha=familywise_alpha, declared_shapes=shapes)

# Calculate the simulator QUOPS/s proxy using summed BR/RR time and shots.
quops_rate_for_each_shape = calculate_quops_rate(data, gamma_for_each_shape)

# Plot every shape's Hochberg decision and its simulator rate side by side.
fig = plot_quops_scan(hochberg_for_each_shape, quops_rate_for_each_shape, title="QUOPS: tested shapes")


### Historical physical scan


<img src="plots/physical_capability_and_rate_scored.png" alt="Historical QUOPS simulation scan through 30 qubits, using the earlier analysis" width="1300">
